In [7]:
using PowerModels, Ipopt, JuMP, DelimitedFiles, Random, Statistics, DataFrames, CSV, LinearAlgebra, SparseArrays, ProgressMeter

function DCOPF_Constraints(case_path::String)
    data = parse_file(case_path)
    ref = PowerModels.build_ref(data)[:it][:pm][:nw][0]

    gen_ids = sort(collect(keys(ref[:gen])))
    branch_ids = sort(collect(keys(ref[:branch])))
    bus_ids = sort(collect(keys(ref[:bus])))
    bus_lookup = Dict(bus_id => i for (i, bus_id) in enumerate(bus_ids))
    bus_count = length(bus_ids)

    p_min = [ref[:gen][i]["pmin"] for i in gen_ids]
    p_max = [ref[:gen][i]["pmax"] for i in gen_ids]
    f_max = [get(ref[:branch][i], "rate_a", Inf) for i in branch_ids]

    cost_c2, cost_c1, cost_c0 = [], [], []
    for gen_id in gen_ids
        cost_coeffs = ref[:gen][gen_id]["cost"]
        if length(cost_coeffs) == 3
            push!(cost_c2, cost_coeffs[1]); push!(cost_c1, cost_coeffs[2]); push!(cost_c0, cost_coeffs[3])
        elseif length(cost_coeffs) == 2
            push!(cost_c2, 0.0); push!(cost_c1, cost_coeffs[1]); push!(cost_c0, cost_coeffs[2])
        else
            push!(cost_c2, 0.0); push!(cost_c1, 0.0); push!(cost_c0, 0.0)
        end
    end

    Bbus = zeros(bus_count, bus_count)
    for (i, branch) in ref[:branch]
        f_bus, t_bus = bus_lookup[branch["f_bus"]], bus_lookup[branch["t_bus"]]
        b = -1 / branch["br_x"]; Bbus[f_bus, t_bus]+=b; Bbus[t_bus, f_bus]+=b; Bbus[f_bus, f_bus]-=b; Bbus[t_bus, t_bus]-=b
    end
    for (i, bus) in ref[:bus]; Bbus[bus_lookup[i], bus_lookup[i]] += get(bus, "bs", 0.0); end

    branch_count = length(branch_ids); A = spzeros(Int, branch_count, bus_count); b_diag = spzeros(Float64, branch_count, branch_count)
    for (i, br_id) in enumerate(branch_ids); branch=ref[:branch][br_id]; f_bus,t_bus=bus_lookup[branch["f_bus"]],bus_lookup[branch["t_bus"]]; A[i,f_bus]=1; A[i,t_bus]=-1; b_diag[i,i]=-1/branch["br_x"]; end

    slack_bus_idx = bus_lookup[first(collect(keys(ref[:ref_buses])))]
    non_slack_indices = [i for i in 1:bus_count if i != slack_bus_idx]
    Bbus_ns = Bbus[non_slack_indices, non_slack_indices]; A_ns = A[:, non_slack_indices]
    ptdf_matrix_ns = b_diag * A_ns * inv(Matrix(Bbus_ns))
    ptdf_matrix = zeros(branch_count, bus_count); ptdf_matrix[:, non_slack_indices] = ptdf_matrix_ns

    gen_count = length(gen_ids); bus_gen_map = zeros(Int, bus_count, gen_count)
    for (i, gen_id) in enumerate(gen_ids); gen_bus = ref[:gen][gen_id]["gen_bus"]; bus_pos = bus_lookup[gen_bus]; bus_gen_map[bus_pos, i] = 1; end

    return ( p_min=p_min, p_max=p_max, f_max=f_max, cost_c1=cost_c1, cost_c2=cost_c2, cost_c0=cost_c0,
             ptdf=ptdf_matrix, bus_gen_map=bus_gen_map, gen_ids=gen_ids, bus_ids=bus_ids,
             branch_ids=branch_ids, bus_lookup=bus_lookup )
end

function DCOPF_dataset_with_duals(case_path::String, output_dir::String, num_samples::Int, variance::Float64, seed::Int; save_intermediate_files::Bool=false)
    Random.seed!(seed)
    println("Generating dataset for $(basename(case_path))")
    println("Target samples: $(num_samples), Variance: $(variance), Seed: $(seed)")

    params = DCOPF_Constraints(case_path)
    
    data = parse_file(case_path)
    base_loads = Dict(load["load_bus"] => load["pd"] for (load_idx, load) in data["load"])
    load_bus_indices = sort(collect(keys(base_loads)))
    
    valid_branch_indices = findall(params.f_max .< 1e10)
    ptdf_constrained = params.ptdf[valid_branch_indices, :]
    f_max_constrained = params.f_max[valid_branch_indices]
    
    num_gens = length(params.gen_ids); num_buses = length(params.bus_ids); num_branches_constrained = length(f_max_constrained)
    
    model = Model(Ipopt.Optimizer); set_silent(model)
    
    @variable(model, pg[1:num_gens]); @variable(model, pd[1:num_buses])
    @objective(model, Min, sum(params.cost_c2[i]*pg[i]^2 + params.cost_c1[i]*pg[i] + params.cost_c0[i] for i in 1:num_gens))
    @constraint(model, power_balance, sum(pg) == sum(pd))
    @constraint(model, gen_min[i=1:num_gens], pg[i] >= params.p_min[i])
    @constraint(model, gen_max[i=1:num_gens], pg[i] <= params.p_max[i])
    @expression(model, pinj[j=1:num_buses], sum(params.bus_gen_map[j, k] * pg[k] for k in 1:num_gens) - pd[j])
    @constraint(model, line_pos[i=1:num_branches_constrained], sum(ptdf_constrained[i, j] * pinj[j] for j in 1:num_buses) <= f_max_constrained[i])
    @constraint(model, line_neg[i=1:num_branches_constrained], sum(ptdf_constrained[i, j] * pinj[j] for j in 1:num_buses) >= -f_max_constrained[i])

    load_data_list, generation_data_list = [], []
    lambda_list, cost_list = [], []
    mu_g_min_list, mu_g_max_list = [], []
    mu_line_pos_list, mu_line_neg_list = [], []

    total_loop_time = @elapsed begin
        @showprogress "Generating Samples" for _ in 1:num_samples
            new_loads = Dict{Int, Float64}()
            for bus_idx in load_bus_indices
                base_pd = base_loads[bus_idx]
                sigma = abs(base_pd * variance)
                new_loads[bus_idx] = max(0.0, base_pd + sigma * randn())
            end

            current_loads = zeros(num_buses)
            for (bus_id, load_val) in new_loads
                bus_pos = params.bus_lookup[bus_id]
                current_loads[bus_pos] = load_val
            end
            for j in 1:num_buses; fix(pd[j], current_loads[j]); end
            
            try
                optimize!(model)
                if termination_status(model) in (MOI.LOCALLY_SOLVED, MOI.OPTIMAL)
                    push!(cost_list, objective_value(model))
                    push!(load_data_list, [new_loads[i] for i in load_bus_indices])
                    push!(generation_data_list, value.(pg))
                    push!(lambda_list, dual(power_balance))
                    push!(mu_g_min_list, dual.(gen_min))
                    push!(mu_g_max_list, -dual.(gen_max))
                    push!(mu_line_pos_list, -dual.(line_pos))
                    push!(mu_line_neg_list, dual.(line_neg))
                end
            catch e
            end
        end 
    end 
    successful_samples = length(cost_list)
    println("Successfully generated $(successful_samples) / $(num_samples) samples in $(round(total_loop_time, digits=2))s.")
    if successful_samples > 0
        println("Average cost: ", round(mean(cost_list), digits=4), "\$")
        println("Average solve time: ", round((total_loop_time / successful_samples) * 1000, digits=2), "ms")
    end

    mkpath(output_dir)
    case_name = split(basename(case_path), ".")[1]
    
    df_loads = DataFrame(hcat(load_data_list...)', Symbol.("pd" .* string.(load_bus_indices)))
    df_gens = DataFrame(hcat(generation_data_list...)', Symbol.("pg" .* string.(params.gen_ids)))
    df_lambda = DataFrame(lambda = lambda_list)
    df_mu_g_min = DataFrame(hcat(mu_g_min_list...)', Symbol.("mu_g_min_" .* string.(params.gen_ids)))
    df_mu_g_max = DataFrame(hcat(mu_g_max_list...)', Symbol.("mu_g_max_" .* string.(params.gen_ids)))
    
    df_mu_line_pos = DataFrame(hcat(mu_line_pos_list...)', Symbol.("mu_line_max_" .* string.(params.branch_ids[valid_branch_indices])))
    df_mu_line_neg = DataFrame(hcat(mu_line_neg_list...)', Symbol.("mu_line_min_" .* string.(params.branch_ids[valid_branch_indices])))

    if save_intermediate_files
        CSV.write(joinpath(output_dir, "$(case_name)_loads.csv"), df_loads)
        CSV.write(joinpath(output_dir, "$(case_name)_generations.csv"), df_gens)
    end
    
    final_df = hcat(df_loads, df_gens, df_lambda, df_mu_g_min, df_mu_g_max, df_mu_line_pos, df_mu_line_neg)
    final_dataset_path = joinpath(output_dir, "$(case_name)_dataset_with_duals.csv")
    CSV.write(final_dataset_path, final_df)
    
    println("Dataset saved to: $(final_dataset_path)")
end

function main()
    ROOT_DIR = raw"C:\Users\Aloha\Desktop\dataset"
    CASE_NAME_FULL = "pglib_opf_case300_ieee"
    CASE_NAME_SHORT = "case300"
    VARIANCE_LABEL = "v=0.09" 

    NUM_SAMPLES = 12000
    VARIANCE = 0.09
    SEED = 42
    SAVE_INTERMEDIATE_FILES = true  # Set to true to save separate files for loads, gens, etc.

    case_file = joinpath(ROOT_DIR, "PGlib", "standard", "$(CASE_NAME_FULL).m")
    output_dir = joinpath(ROOT_DIR, "PINN", "$(CASE_NAME_SHORT)($(VARIANCE_LABEL))")

    DCOPF_dataset_with_duals(case_file, output_dir, NUM_SAMPLES, VARIANCE, SEED, save_intermediate_files=SAVE_INTERMEDIATE_FILES)
end

# main
main()

Generating dataset for pglib_opf_case300_ieee.m
Target samples: 12000, Variance: 0.09, Seed: 42
[info | PowerModels]: removing 1 cost terms from generator 32: [3101.6664, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 29: [2727.8538000000003, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 1: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 54: [10398.7496, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 2: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 41: [3965.7688, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 65: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 51: [3512.6651, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 53: [2999.2014999999997, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 27: [2839.1517, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 42: [4122.3312, 0.0]
[info | PowerModels]: removi

Generating Samples 100%|█████████████████████████████████| Time: 4:46:23


Successfully generated 8842 / 12000 samples in 17183.73s.
Average cost: 526337.0739$
Average solve time: 1943.42ms
Dataset saved to: C:\Users\Aloha\Desktop\dataset\PINN\case300(v=0.09)\pglib_opf_case300_ieee_dataset_with_duals.csv


In [11]:
using PowerModels, Ipopt, JuMP, DelimitedFiles, Random, Statistics, DataFrames, CSV, LinearAlgebra, SparseArrays, ProgressMeter

"""
    DCOPF_Constraints(case_path::String)

从一个PowerModels的案例文件(.m)中提取DC-OPF所需的所用物理参数。
"""
function DCOPF_Constraints(case_path::String)
    data = parse_file(case_path)
    ref = PowerModels.build_ref(data)[:it][:pm][:nw][0]

    gen_ids = sort(collect(keys(ref[:gen])))
    branch_ids = sort(collect(keys(ref[:branch])))
    bus_ids = sort(collect(keys(ref[:bus])))
    bus_lookup = Dict(bus_id => i for (i, bus_id) in enumerate(bus_ids))
    bus_count = length(bus_ids)

    p_min = [ref[:gen][i]["pmin"] for i in gen_ids]
    p_max = [ref[:gen][i]["pmax"] for i in gen_ids]
    f_max = [get(ref[:branch][i], "rate_a", Inf) for i in branch_ids]

    cost_c2, cost_c1, cost_c0 = [], [], []
    for gen_id in gen_ids
        cost_coeffs = ref[:gen][gen_id]["cost"]
        if length(cost_coeffs) == 3; push!(cost_c2, cost_coeffs[1]); push!(cost_c1, cost_coeffs[2]); push!(cost_c0, cost_coeffs[3]);
        elseif length(cost_coeffs) == 2; push!(cost_c2, 0.0); push!(cost_c1, cost_coeffs[1]); push!(cost_c0, cost_coeffs[2]);
        else; push!(cost_c2, 0.0); push!(cost_c1, 0.0); push!(cost_c0, 0.0); end
    end

    Bbus = zeros(bus_count, bus_count)
    for (i, branch) in ref[:branch]; f_bus, t_bus = bus_lookup[branch["f_bus"]], bus_lookup[branch["t_bus"]]; b = -1 / branch["br_x"]; Bbus[f_bus, t_bus]+=b; Bbus[t_bus, f_bus]+=b; Bbus[f_bus, f_bus]-=b; Bbus[t_bus, t_bus]-=b; end
    for (i, bus) in ref[:bus]; Bbus[bus_lookup[i], bus_lookup[i]] += get(bus, "bs", 0.0); end

    branch_count = length(branch_ids); A = spzeros(Int, branch_count, bus_count); b_diag = spzeros(Float64, branch_count, branch_count)
    for (i, br_id) in enumerate(branch_ids); branch=ref[:branch][br_id]; f_bus,t_bus=bus_lookup[branch["f_bus"]],bus_lookup[branch["t_bus"]]; A[i,f_bus]=1; A[i,t_bus]=-1; b_diag[i,i]=-1/branch["br_x"]; end

    slack_bus_idx = bus_lookup[first(collect(keys(ref[:ref_buses])))]
    non_slack_indices = [i for i in 1:bus_count if i != slack_bus_idx]
    Bbus_ns = Bbus[non_slack_indices, non_slack_indices]; A_ns = A[:, non_slack_indices]
    ptdf_matrix_ns = b_diag * A_ns * inv(Matrix(Bbus_ns))
    ptdf_matrix = zeros(branch_count, bus_count); ptdf_matrix[:, non_slack_indices] = ptdf_matrix_ns

    gen_count = length(gen_ids); bus_gen_map = zeros(Int, bus_count, gen_count)
    for (i, gen_id) in enumerate(gen_ids); gen_bus = ref[:gen][gen_id]["gen_bus"]; bus_pos = bus_lookup[gen_bus]; bus_gen_map[bus_pos, i] = 1; end

    return ( p_min=p_min, p_max=p_max, f_max=f_max, cost_c1=cost_c1, cost_c2=cost_c2, cost_c0=cost_c0,
             ptdf=ptdf_matrix, bus_gen_map=bus_gen_map, gen_ids=gen_ids, bus_ids=bus_ids,
             branch_ids=branch_ids, bus_lookup=bus_lookup )
end

"""
    DCOPF_dataset_with_duals(...)

通过对基准负荷施加正态分布扰动来生成DC-OPF数据集，并包含所有主变量和对偶变量。
"""
function DCOPF_dataset_with_duals(case_path::String, output_dir::String, num_samples::Int, variance::Float64, seed::Int; save_intermediate_files::Bool=false)
    Random.seed!(seed)
    println("Generating DC-OPF dataset for $(basename(case_path))")
    println("Target samples: $(num_samples), Variance: $(variance), Seed: $(seed)")

    params = DCOPF_Constraints(case_path)
    
    data = parse_file(case_path)
    base_loads = Dict(load["load_bus"] => load["pd"] for (load_idx, load) in data["load"])
    load_bus_indices = sort(collect(keys(base_loads)))
    
    valid_branch_indices = findall(params.f_max .< 1e10)
    ptdf_constrained = params.ptdf[valid_branch_indices, :]
    f_max_constrained = params.f_max[valid_branch_indices]
    
    num_gens = length(params.gen_ids); num_buses = length(params.bus_ids)
    
    model = Model(Ipopt.Optimizer); set_silent(model)
    
    @variable(model, pg[1:num_gens]); @variable(model, pd[1:num_buses])
    @objective(model, Min, sum(params.cost_c2[i]*pg[i]^2 + params.cost_c1[i]*pg[i] + params.cost_c0[i] for i in 1:num_gens))
    @constraint(model, power_balance, sum(pg) == sum(pd))
    @constraint(model, gen_min[i=1:num_gens], pg[i] >= params.p_min[i])
    @constraint(model, gen_max[i=1:num_gens], pg[i] <= params.p_max[i])
    @expression(model, pinj[j=1:num_buses], sum(params.bus_gen_map[j, k] * pg[k] for k in 1:num_gens) - pd[j])
    @constraint(model, line_pos[i in 1:length(valid_branch_indices)], sum(ptdf_constrained[i, j] * pinj[j] for j in 1:num_buses) <= f_max_constrained[i])
    @constraint(model, line_neg[i in 1:length(valid_branch_indices)], sum(ptdf_constrained[i, j] * pinj[j] for j in 1:num_buses) >= -f_max_constrained[i])

    results = []
    first_sample_diagnosed = false

    total_loop_time = @elapsed begin
        @showprogress "Generating Samples" for _ in 1:num_samples
            new_loads = Dict{Int, Float64}()
            for bus_idx in load_bus_indices
                new_loads[bus_idx] = max(0.0, base_loads[bus_idx] + abs(base_loads[bus_idx] * variance) * randn())
            end

            current_loads = zeros(num_buses)
            for (bus_id, load_val) in new_loads; current_loads[params.bus_lookup[bus_id]] = load_val; end
            for j in 1:num_buses; fix(pd[j], current_loads[j]); end
            
            try
                optimize!(model)
                if termination_status(model) in (MOI.LOCALLY_SOLVED, MOI.OPTIMAL)
                    pg_solution = value.(pg)
                    mu_g_max_solution = -dual.(gen_max)
                    
                    # --- **内置诊断功能 (已更新)** ---
                    if !first_sample_diagnosed
                        println("\n\n--- 正在诊断第一个成功的DC-OPF样本 ---")
                        
                        # 遍历所有发电机并打印其状态
                        for i in 1:num_gens
                            gen_id_int = params.gen_ids[i]
                            pg_val = pg_solution[i]
                            p_max_limit = params.p_max[i]
                            mu_val = mu_g_max_solution[i]
                            gap = p_max_limit - pg_val
                            
                            println("\n[发电机 ID: $(gen_id_int)]")
                            println("  有功出力上限 (Pmax): $(p_max_limit)")
                            println("  实际有功出力 (PG):  $(round(pg_val, digits=4))")
                            println("  >> 差距 (上限 - 实际): $(round(gap, digits=4))")
                            
                            # 只有当差距非常小（约束被激活）时，mu才应该非零
                            if abs(gap) < 1e-4 && mu_val > 1e-6
                                println("  状态: \e[31m已激活 (Active)\e[0m") # Red color for active
                                println("  因此，mu_pg_max 被正确地计算为: \e[31m$(round(mu_val, digits=4)) (非零!)\e[0m")
                            else
                                println("  状态: 未激活 (Inactive)")
                                println("  因此，mu_pg_max 被正确地计算为: $(round(mu_val, digits=4))")
                            end
                        end
                        
                        println("\n--- 诊断结束 ---\n")
                        first_sample_diagnosed = true
                    end
                    # --- 诊断结束 ---
                    
                    push!(results, (
                        loads=[new_loads[i] for i in load_bus_indices],
                        pg=pg_solution,
                        lambda=dual(power_balance),
                        mu_g_min=dual.(gen_min),
                        mu_g_max=mu_g_max_solution,
                        mu_line_pos=-dual.(line_pos),
                        mu_line_neg=dual.(line_neg),
                        cost=objective_value(model)
                    ))
                end
            catch e
            end
        end 
    end 
    
    successful_samples = length(results)
    println("成功生成 $(successful_samples) / $(num_samples) 个样本，耗时 $(round(total_loop_time, digits=2))秒。")
    if successful_samples > 0
        println("平均成本: ", round(mean([r.cost for r in results]), digits=4), "\$/h")
        println("平均求解时间: ", round((total_loop_time / successful_samples) * 1000, digits=2), "ms")
    end

    mkpath(output_dir)
    case_name = split(basename(case_path), ".")[1]
    
    df_loads = DataFrame(hcat([r.loads for r in results]...)' , Symbol.("pd" .* string.(load_bus_indices)))
    df_gens = DataFrame(hcat([r.pg for r in results]...)'    , Symbol.("pg" .* string.(params.gen_ids)))
    df_lambda = DataFrame(lambda = [r.lambda for r in results])
    df_mu_g_min = DataFrame(hcat([r.mu_g_min for r in results]...)' , Symbol.("mu_g_min_" .* string.(params.gen_ids)))
    df_mu_g_max = DataFrame(hcat([r.mu_g_max for r in results]...)' , Symbol.("mu_g_max_" .* string.(params.gen_ids)))
    
    df_mu_line_pos = DataFrame(hcat([r.mu_line_pos for r in results]...)' , Symbol.("mu_line_max_" .* string.(params.branch_ids[valid_branch_indices])))
    df_mu_line_neg = DataFrame(hcat([r.mu_line_neg for r in results]...)' , Symbol.("mu_line_min_" .* string.(params.branch_ids[valid_branch_indices])))

    if save_intermediate_files
        CSV.write(joinpath(output_dir, "$(case_name)_loads.csv"), df_loads)
        CSV.write(joinpath(output_dir, "$(case_name)_generations.csv"), df_gens)
    end
    
    final_df = hcat(df_loads, df_gens, df_lambda, df_mu_g_min, df_mu_g_max, df_mu_line_pos, df_mu_line_neg)
    final_dataset_path = joinpath(output_dir, "$(case_name)_dataset_with_duals.csv")
    CSV.write(final_dataset_path, final_df)
    
    println("数据集已保存至: $(final_dataset_path)")
end

# ===================================================================
# --- 主配置和执行区 ---
# ===================================================================
function main()
    # 1. 配置
    ROOT_DIR = raw"C:\Users\Aloha\Desktop\dataset"
    CASE_NAME_FULL = "pglib_opf_case14_ieee"
    CASE_NAME_SHORT = "case14"
    VARIANCE_LABEL = "v=0.12" 

    NUM_SAMPLES = 50000
    VARIANCE = 0.12
    SEED = 42
    SAVE_INTERMEDIATE_FILES = true

    # 2. 动态路径构建
    case_file = joinpath(ROOT_DIR, "PGlib", "standard", "$(CASE_NAME_FULL).m")
    output_dir = joinpath(ROOT_DIR, "PINN", "$(CASE_NAME_SHORT)($(VARIANCE_LABEL))_diag_all_gens")

    # 3. 执行
    DCOPF_dataset_with_duals(case_file, output_dir, NUM_SAMPLES, VARIANCE, SEED, save_intermediate_files=SAVE_INTERMEDIATE_FILES)
end

main()

Generating DC-OPF dataset for pglib_opf_case14_ieee.m
Target samples: 1000, Variance: 0.12, Seed: 42
[info | PowerModels]: removing 3 cost terms from generator 4: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 1: [792.0951, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 5: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 2: [2326.9494, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 3: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 4: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 1: [792.0951, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 5: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 2: [2326.9494, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 3: Float64[]


Generating Samples   0%|█                                |  ETA: 0:00:20



--- 正在诊断第一个成功的DC-OPF样本 ---

[发电机 ID: 1]
  有功出力上限 (Pmax): 3.4
  实际有功出力 (PG):  2.4541
  >> 差距 (上限 - 实际): 0.9459
  状态: 未激活 (Inactive)
  因此，mu_pg_max 被正确地计算为: 0.0

[发电机 ID: 2]
  有功出力上限 (Pmax): 0.59
  实际有功出力 (PG):  -0.0
  >> 差距 (上限 - 实际): 0.59
  状态: 未激活 (Inactive)
  因此，mu_pg_max 被正确地计算为: 0.0

[发电机 ID: 3]
  有功出力上限 (Pmax): 0.0
  实际有功出力 (PG):  -0.0
  >> 差距 (上限 - 实际): 0.0
  状态: 已激活 (Active)
  因此，mu_pg_max 被正确地计算为: 4.22485410445e7 (非零!)

[发电机 ID: 4]
  有功出力上限 (Pmax): 0.0
  实际有功出力 (PG):  -0.0
  >> 差距 (上限 - 实际): 0.0
  状态: 已激活 (Active)
  因此，mu_pg_max 被正确地计算为: 4.07724338206e7 (非零!)

[发电机 ID: 5]
  有功出力上限 (Pmax): 0.0
  实际有功出力 (PG):  0.0
  >> 差距 (上限 - 实际): -0.0
  状态: 已激活 (Active)
  因此，mu_pg_max 被正确地计算为: 4.09903341486e7 (非零!)

--- 诊断结束 ---



Generating Samples 100%|█████████████████████████████████| Time: 0:00:12

成功生成 998 / 1000 个样本，耗时 12.24秒。
平均成本: 2048.9454$/h
平均求解时间: 12.26ms


数据集已保存至: C:\Users\Aloha\Desktop\dataset\PINN\case14(v=0.12)_diag_all_gens\pglib_opf_case14_ieee_dataset_with_duals.csv


In [ ]:
using PowerModels, Ipopt, JuMP, DelimitedFiles, Random, Statistics, DataFrames, CSV, LinearAlgebra, SparseArrays, ProgressMeter

function DCOPF_Constraints(case_path::String)
    data = parse_file(case_path)
    ref = PowerModels.build_ref(data)[:it][:pm][:nw][0]

    gen_ids = sort(collect(keys(ref[:gen])))
    branch_ids = sort(collect(keys(ref[:branch])))
    bus_ids = sort(collect(keys(ref[:bus])))
    bus_lookup = Dict(bus_id => i for (i, bus_id) in enumerate(bus_ids))
    bus_count = length(bus_ids)

    p_min = [ref[:gen][i]["pmin"] for i in gen_ids]
    p_max = [ref[:gen][i]["pmax"] for i in gen_ids]
    f_max = [get(ref[:branch][i], "rate_a", Inf) for i in branch_ids]

    cost_c2, cost_c1, cost_c0 = [], [], []
    for gen_id in gen_ids
        cost_coeffs = ref[:gen][gen_id]["cost"]
        if length(cost_coeffs) == 3
            push!(cost_c2, cost_coeffs[1]); push!(cost_c1, cost_coeffs[2]); push!(cost_c0, cost_coeffs[3])
        elseif length(cost_coeffs) == 2
            push!(cost_c2, 0.0); push!(cost_c1, cost_coeffs[1]); push!(cost_c0, cost_coeffs[2])
        else
            push!(cost_c2, 0.0); push!(cost_c1, 0.0); push!(cost_c0, 0.0)
        end
    end

    Bbus = zeros(bus_count, bus_count)
    for (i, branch) in ref[:branch]
        f_bus, t_bus = bus_lookup[branch["f_bus"]], bus_lookup[branch["t_bus"]]
        b = -1 / branch["br_x"]; Bbus[f_bus, t_bus]+=b; Bbus[t_bus, f_bus]+=b; Bbus[f_bus, f_bus]-=b; Bbus[t_bus, t_bus]-=b
    end
    for (i, bus) in ref[:bus]; Bbus[bus_lookup[i], bus_lookup[i]] += get(bus, "bs", 0.0); end

    branch_count = length(branch_ids); A = spzeros(Int, branch_count, bus_count); b_diag = spzeros(Float64, branch_count, branch_count)
    for (i, br_id) in enumerate(branch_ids); branch=ref[:branch][br_id]; f_bus,t_bus=bus_lookup[branch["f_bus"]],bus_lookup[branch["t_bus"]]; A[i,f_bus]=1; A[i,t_bus]=-1; b_diag[i,i]=-1/branch["br_x"]; end

    slack_bus_idx = bus_lookup[first(collect(keys(ref[:ref_buses])))]
    non_slack_indices = [i for i in 1:bus_count if i != slack_bus_idx]
    Bbus_ns = Bbus[non_slack_indices, non_slack_indices]; A_ns = A[:, non_slack_indices]
    ptdf_matrix_ns = b_diag * A_ns * inv(Matrix(Bbus_ns))
    ptdf_matrix = zeros(branch_count, bus_count); ptdf_matrix[:, non_slack_indices] = ptdf_matrix_ns

    gen_count = length(gen_ids); bus_gen_map = zeros(Int, bus_count, gen_count)
    for (i, gen_id) in enumerate(gen_ids); gen_bus = ref[:gen][gen_id]["gen_bus"]; bus_pos = bus_lookup[gen_bus]; bus_gen_map[bus_pos, i] = 1; end

    return ( p_min=p_min, p_max=p_max, f_max=f_max, cost_c1=cost_c1, cost_c2=cost_c2, cost_c0=cost_c0,
             ptdf=ptdf_matrix, bus_gen_map=bus_gen_map, gen_ids=gen_ids, bus_ids=bus_ids,
             branch_ids=branch_ids, bus_lookup=bus_lookup )
end

function DCOPF_dataset_with_duals(case_path::String, output_dir::String, num_samples::Int, variance::Float64, seed::Int; 
                                  save_intermediate_files::Bool=false, 
                                  sampling_method::String="gaussian", 
                                  sampling_range::Vector{Float64}=[0.9, 1.1])
    Random.seed!(seed)
    println("Generating dataset for $(basename(case_path))")
    println("Target samples: $(num_samples), Sampling: $(sampling_method), Seed: $(seed)")

    params = DCOPF_Constraints(case_path)
    
    data = parse_file(case_path)
    base_loads = Dict(load["load_bus"] => load["pd"] for (load_idx, load) in data["load"])
    load_bus_indices = sort(collect(keys(base_loads)))
    
    valid_branch_indices = findall(params.f_max .< 1e10)
    ptdf_constrained = params.ptdf[valid_branch_indices, :]
    f_max_constrained = params.f_max[valid_branch_indices]
    
    num_gens = length(params.gen_ids); num_buses = length(params.bus_ids); num_branches_constrained = length(f_max_constrained)
    
    model = Model(Ipopt.Optimizer); set_silent(model)
    
    @variable(model, pg[1:num_gens]); @variable(model, pd[1:num_buses])
    @objective(model, Min, sum(params.cost_c2[i]*pg[i]^2 + params.cost_c1[i]*pg[i] + params.cost_c0[i] for i in 1:num_gens))
    @constraint(model, power_balance, sum(pg) == sum(pd))
    @constraint(model, gen_min[i=1:num_gens], pg[i] >= params.p_min[i])
    @constraint(model, gen_max[i=1:num_gens], pg[i] <= params.p_max[i])
    @expression(model, pinj[j=1:num_buses], sum(params.bus_gen_map[j, k] * pg[k] for k in 1:num_gens) - pd[j])
    @constraint(model, line_pos[i=1:num_branches_constrained], sum(ptdf_constrained[i, j] * pinj[j] for j in 1:num_buses) <= f_max_constrained[i])
    @constraint(model, line_neg[i=1:num_branches_constrained], sum(ptdf_constrained[i, j] * pinj[j] for j in 1:num_buses) >= -f_max_constrained[i])

    load_data_list, generation_data_list = [], []
    lambda_list, cost_list = [], []
    mu_g_min_list, mu_g_max_list = [], []
    mu_line_pos_list, mu_line_neg_list = [], []

    total_loop_time = @elapsed begin
        @showprogress "Generating Samples" for _ in 1:num_samples
            new_loads = Dict{Int, Float64}()
            for bus_idx in load_bus_indices
                base_pd = base_loads[bus_idx]
                sampled_pd = 0.0

                # --- MODIFICATION: Switch between sampling methods ---
                if sampling_method == "gaussian"
                    sigma = abs(base_pd * variance)
                    sampled_pd = max(0.0, base_pd + sigma * randn())
                elseif sampling_method == "uniform"
                    lower_bound = base_pd * sampling_range[1]
                    upper_bound = base_pd * sampling_range[2]
                    sampled_pd = lower_bound + (upper_bound - lower_bound) * rand()
                else
                    error("Unsupported sampling method: $(sampling_method). Use 'gaussian' or 'uniform'.")
                end
                new_loads[bus_idx] = sampled_pd
            end

            current_loads = zeros(num_buses)
            for (bus_id, load_val) in new_loads
                bus_pos = params.bus_lookup[bus_id]
                current_loads[bus_pos] = load_val
            end
            for j in 1:num_buses; fix(pd[j], current_loads[j]); end
            
            try
                optimize!(model)
                if termination_status(model) in (MOI.LOCALLY_SOLVED, MOI.OPTIMAL)
                    push!(cost_list, objective_value(model))
                    push!(load_data_list, [new_loads[i] for i in load_bus_indices])
                    push!(generation_data_list, value.(pg))
                    push!(lambda_list, dual(power_balance))
                    push!(mu_g_min_list, dual.(gen_min))
                    push!(mu_g_max_list, -dual.(gen_max))
                    push!(mu_line_pos_list, -dual.(line_pos))
                    push!(mu_line_neg_list, dual.(line_neg))
                end
            catch e
                # Silently ignore solver errors and continue
            end
        end 
    end 
    successful_samples = length(cost_list)
    println("Successfully generated $(successful_samples) / $(num_samples) samples in $(round(total_loop_time, digits=2))s.")
    if successful_samples > 0
        println("Average cost: ", round(mean(cost_list), digits=4), "\$")
        println("Average solve time: ", round((total_loop_time / successful_samples) * 1000, digits=2), "ms")
    end

    mkpath(output_dir)
    case_name = split(basename(case_path), ".")[1]
    
    df_loads = DataFrame(hcat(load_data_list...)', Symbol.("pd" .* string.(load_bus_indices)))
    df_gens = DataFrame(hcat(generation_data_list...)', Symbol.("pg" .* string.(params.gen_ids)))
    df_lambda = DataFrame(lambda = lambda_list)
    df_mu_g_min = DataFrame(hcat(mu_g_min_list...)', Symbol.("mu_g_min_" .* string.(params.gen_ids)))
    df_mu_g_max = DataFrame(hcat(mu_g_max_list...)', Symbol.("mu_g_max_" .* string.(params.gen_ids)))
    
    df_mu_line_pos = DataFrame(hcat(mu_line_pos_list...)', Symbol.("mu_line_max_" .* string.(params.branch_ids[valid_branch_indices])))
    df_mu_line_neg = DataFrame(hcat(mu_line_neg_list...)', Symbol.("mu_line_min_" .* string.(params.branch_ids[valid_branch_indices])))

    if save_intermediate_files
        CSV.write(joinpath(output_dir, "$(case_name)_loads.csv"), df_loads)
        CSV.write(joinpath(output_dir, "$(case_name)_generations.csv"), df_gens)
    end
    
    final_df = hcat(df_loads, df_gens, df_lambda, df_mu_g_min, df_mu_g_max, df_mu_line_pos, df_mu_line_neg)
    final_dataset_path = joinpath(output_dir, "$(case_name)_dataset_with_duals.csv")
    CSV.write(final_dataset_path, final_df)
    
    println("Dataset saved to: $(final_dataset_path)")
end

function main()
    ROOT_DIR = raw"C:\Users\Aloha\Desktop\dataset"
    CASE_NAME_FULL = "pglib_opf_case30_ieee"
    CASE_NAME_SHORT = "case30"
    
    # --- CONFIGURATION ---
    # Choose sampling method: "gaussian" or "uniform"
    SAMPLING_METHOD = "uniform"
    SAMPLING_RANGE = [0.9, 1.1] # [90%, 110%] for uniform sampling

    # This variance is only used for Gaussian sampling, but we define it here
    VARIANCE = 0.12 

    # --- Generate a descriptive label for the output folder ---
    if SAMPLING_METHOD == "uniform"
        lower_pct = Int(SAMPLING_RANGE[1] * 100)
        upper_pct = Int(SAMPLING_RANGE[2] * 100)
        SAMPLING_LABEL = "u=$(lower_pct)-$(upper_pct)"
    else
        SAMPLING_LABEL = "v=$(VARIANCE)"
    end

    NUM_SAMPLES = 50000
    SEED = 42
    SAVE_INTERMEDIATE_FILES = true

    case_file = joinpath(ROOT_DIR, "PGlib", "standard", "$(CASE_NAME_FULL).m")
    output_dir = joinpath(ROOT_DIR, "PINN", "$(CASE_NAME_SHORT)($(SAMPLING_LABEL))")

    DCOPF_dataset_with_duals(
        case_file, 
        output_dir, 
        NUM_SAMPLES, 
        VARIANCE, 
        SEED, 
        save_intermediate_files=SAVE_INTERMEDIATE_FILES,
        sampling_method=SAMPLING_METHOD,
        sampling_range=SAMPLING_RANGE
    )
end

# main
main()

In [7]:
using PowerModels, Ipopt, JuMP, DelimitedFiles, Random, Statistics, DataFrames, CSV, LinearAlgebra, SparseArrays, ProgressMeter

function DCOPF_Constraints(case_path::String)
    data = parse_file(case_path)
    ref = PowerModels.build_ref(data)[:it][:pm][:nw][0]

    gen_ids = sort(collect(keys(ref[:gen])))
    branch_ids = sort(collect(keys(ref[:branch])))
    bus_ids = sort(collect(keys(ref[:bus])))
    bus_lookup = Dict(bus_id => i for (i, bus_id) in enumerate(bus_ids))
    bus_count = length(bus_ids)

    p_min = [ref[:gen][i]["pmin"] for i in gen_ids]
    p_max = [ref[:gen][i]["pmax"] for i in gen_ids]
    f_max = [get(ref[:branch][i], "rate_a", Inf) for i in branch_ids]

    cost_c2, cost_c1, cost_c0 = [], [], []
    for gen_id in gen_ids
        cost_coeffs = ref[:gen][gen_id]["cost"]
        if length(cost_coeffs) == 3
            push!(cost_c2, cost_coeffs[1]); push!(cost_c1, cost_coeffs[2]); push!(cost_c0, cost_coeffs[3])
        elseif length(cost_coeffs) == 2
            push!(cost_c2, 0.0); push!(cost_c1, cost_coeffs[1]); push!(cost_c0, cost_coeffs[2])
        else
            push!(cost_c2, 0.0); push!(cost_c1, 0.0); push!(cost_c0, 0.0)
        end
    end

    Bbus = zeros(bus_count, bus_count)
    for (i, branch) in ref[:branch]
        f_bus, t_bus = bus_lookup[branch["f_bus"]], bus_lookup[branch["t_bus"]]
        b = -1 / branch["br_x"]; Bbus[f_bus, t_bus]+=b; Bbus[t_bus, f_bus]+=b; Bbus[f_bus, f_bus]-=b; Bbus[t_bus, t_bus]-=b
    end
    for (i, bus) in ref[:bus]; Bbus[bus_lookup[i], bus_lookup[i]] += get(bus, "bs", 0.0); end

    branch_count = length(branch_ids); A = spzeros(Int, branch_count, bus_count); b_diag = spzeros(Float64, branch_count, branch_count)
    for (i, br_id) in enumerate(branch_ids); branch=ref[:branch][br_id]; f_bus,t_bus=bus_lookup[branch["f_bus"]],bus_lookup[branch["t_bus"]]; A[i,f_bus]=1; A[i,t_bus]=-1; b_diag[i,i]=-1/branch["br_x"]; end

    slack_bus_idx = bus_lookup[first(collect(keys(ref[:ref_buses])))]
    non_slack_indices = [i for i in 1:bus_count if i != slack_bus_idx]
    Bbus_ns = Bbus[non_slack_indices, non_slack_indices]; A_ns = A[:, non_slack_indices]
    ptdf_matrix_ns = b_diag * A_ns * inv(Matrix(Bbus_ns))
    ptdf_matrix = zeros(branch_count, bus_count); ptdf_matrix[:, non_slack_indices] = ptdf_matrix_ns

    gen_count = length(gen_ids); bus_gen_map = zeros(Int, bus_count, gen_count)
    for (i, gen_id) in enumerate(gen_ids); gen_bus = ref[:gen][gen_id]["gen_bus"]; bus_pos = bus_lookup[gen_bus]; bus_gen_map[bus_pos, i] = 1; end

    return ( p_min=p_min, p_max=p_max, f_max=f_max, cost_c1=cost_c1, cost_c2=cost_c2, cost_c0=cost_c0,
             ptdf=ptdf_matrix, bus_gen_map=bus_gen_map, gen_ids=gen_ids, bus_ids=bus_ids,
             branch_ids=branch_ids, bus_lookup=bus_lookup )
end

function DCOPF_dataset_with_duals(case_path::String, output_dir::String, num_samples::Int, variance::Float64, seed::Int; save_intermediate_files::Bool=false)
    Random.seed!(seed)
    println("Generating dataset for $(basename(case_path))")
    println("Target samples: $(num_samples), Variance: $(variance), Seed: $(seed)")

    params = DCOPF_Constraints(case_path)
    
    data = parse_file(case_path)
    base_loads = Dict(load["load_bus"] => load["pd"] for (load_idx, load) in data["load"])
    load_bus_indices = sort(collect(keys(base_loads)))
    
    valid_branch_indices = findall(params.f_max .< 1e10)
    ptdf_constrained = params.ptdf[valid_branch_indices, :]
    f_max_constrained = params.f_max[valid_branch_indices]
    
    num_gens = length(params.gen_ids); num_buses = length(params.bus_ids); num_branches_constrained = length(f_max_constrained)
    
    model = Model(Ipopt.Optimizer); set_silent(model)
    
    @variable(model, pg[1:num_gens]); @variable(model, pd[1:num_buses])
    @objective(model, Min, sum(params.cost_c2[i]*pg[i]^2 + params.cost_c1[i]*pg[i] + params.cost_c0[i] for i in 1:num_gens))
    
    @constraint(model, power_balance, sum(pg) == sum(pd))
    @constraint(model, gen_min[i=1:num_gens], pg[i] >= params.p_min[i])
    @constraint(model, gen_max[i=1:num_gens], pg[i] <= params.p_max[i])
    @expression(model, pinj[j=1:num_buses], sum(params.bus_gen_map[j, k] * pg[k] for k in 1:num_gens) - pd[j])
    @constraint(model, line_pos[i=1:num_branches_constrained], sum(ptdf_constrained[i, j] * pinj[j] for j in 1:num_buses) <= f_max_constrained[i])
    @constraint(model, line_neg[i=1:num_branches_constrained], sum(ptdf_constrained[i, j] * pinj[j] for j in 1:num_buses) >= -f_max_constrained[i])

    load_data_list, generation_data_list = [], []
    lambda_list, cost_list = [], []
    mu_g_min_list, mu_g_max_list = [], []
    mu_line_pos_list, mu_line_neg_list = [], []

    total_loop_time = @elapsed begin
        @showprogress "Generating Samples" for _ in 1:num_samples
            new_loads = Dict{Int, Float64}()
            for bus_idx in load_bus_indices
                base_pd = base_loads[bus_idx]
                sigma = abs(base_pd * variance)
                new_loads[bus_idx] = max(0.0, base_pd + sigma * randn())
            end

            current_loads = zeros(num_buses)
            for (bus_id, load_val) in new_loads
                bus_pos = params.bus_lookup[bus_id]
                current_loads[bus_pos] = load_val
            end
            for j in 1:num_buses; fix(pd[j], current_loads[j]); end
            
            try
                optimize!(model)
                if termination_status(model) in (MOI.LOCALLY_SOLVED, MOI.OPTIMAL)
                    push!(cost_list, objective_value(model))
                    push!(load_data_list, [new_loads[i] for i in load_bus_indices])
                    push!(generation_data_list, value.(pg))
                    
                    # ===== 🔧 正确方法：从线路流对偶变量计算每个母线的LMP =====
                    lambda_sys = dual(power_balance)  # 系统边际成本
                    mu_line_pos = -dual.(line_pos)   # 线路上限对偶变量（注意符号）
                    mu_line_neg = dual.(line_neg)     # 线路下限对偶变量
                    
                    # 计算每个母线的LMP
                    # LMP[j] = lambda_sys + sum(PTDF[i,j] * (mu_line_pos[i] - mu_line_neg[i]))
                    nodal_lmps = zeros(num_buses)
                    for j in 1:num_buses
                        nodal_lmps[j] = lambda_sys
                        for i in 1:num_branches_constrained
                            nodal_lmps[j] += ptdf_constrained[i, j] * (mu_line_pos[i] - mu_line_neg[i])
                        end
                    end
                    
                    push!(lambda_list, nodal_lmps)
                    push!(mu_g_min_list, dual.(gen_min))
                    push!(mu_g_max_list, -dual.(gen_max))
                    push!(mu_line_pos_list, mu_line_pos)
                    push!(mu_line_neg_list, mu_line_neg)
                end
            catch e
                # 如果求解失败，跳过这个样本
                println("Sample failed: ", e)
            end
        end 
    end 
    
    successful_samples = length(cost_list)
    println("Successfully generated $(successful_samples) / $(num_samples) samples in $(round(total_loop_time, digits=2))s.")
    if successful_samples > 0
        println("Average cost: ", round(mean(cost_list), digits=4), "\$")
        println("Average solve time: ", round((total_loop_time / successful_samples) * 1000, digits=2), "ms")
        
        # ===== 显示LMP统计信息 =====
        lmp_matrix = hcat(lambda_list...)'  # 转换为矩阵 (n_samples × n_buses)
        println("\n📊 LMP Statistics (节点电价统计):")
        for bus_idx in 1:num_buses
            bus_lmps = lmp_matrix[:, bus_idx]
            println("  Bus $(params.bus_ids[bus_idx]): μ ∈ [$(round(minimum(bus_lmps), digits=4)), $(round(maximum(bus_lmps), digits=4))], " *
                    "mean=$(round(mean(bus_lmps), digits=4)), std=$(round(std(bus_lmps), digits=4))")
        end
    else
        error("No successful samples generated! Check your optimization setup.")
    end

    mkpath(output_dir)
    case_name = split(basename(case_path), ".")[1]
    
    df_loads = DataFrame(hcat(load_data_list...)', Symbol.("pd" .* string.(load_bus_indices)))
    df_gens = DataFrame(hcat(generation_data_list...)', Symbol.("pg" .* string.(params.gen_ids)))
    
    # ===== 保存每个母线的LMP=====
    df_lambda = DataFrame(
        hcat(lambda_list...)', 
        Symbol.("lambda" .* string.(params.bus_ids))
    )
    
    df_mu_g_min = DataFrame(hcat(mu_g_min_list...)', Symbol.("mu_g_min_" .* string.(params.gen_ids)))
    df_mu_g_max = DataFrame(hcat(mu_g_max_list...)', Symbol.("mu_g_max_" .* string.(params.gen_ids)))
    
    df_mu_line_pos = DataFrame(hcat(mu_line_pos_list...)', Symbol.("mu_line_max_" .* string.(params.branch_ids[valid_branch_indices])))
    df_mu_line_neg = DataFrame(hcat(mu_line_neg_list...)', Symbol.("mu_line_min_" .* string.(params.branch_ids[valid_branch_indices])))

    if save_intermediate_files
        CSV.write(joinpath(output_dir, "$(case_name)_loads.csv"), df_loads)
        CSV.write(joinpath(output_dir, "$(case_name)_generations.csv"), df_gens)
        CSV.write(joinpath(output_dir, "$(case_name)_nodal_lmps.csv"), df_lambda)
    end
    
    final_df = hcat(df_loads, df_gens, df_lambda, df_mu_g_min, df_mu_g_max, df_mu_line_pos, df_mu_line_neg)
    final_dataset_path = joinpath(output_dir, "$(case_name)_dataset_with_duals.csv")
    CSV.write(final_dataset_path, final_df)
    
    println("\n✅ Dataset saved to: $(final_dataset_path)")
    println("   Total columns: $(ncol(final_df))")
    println("   - Load columns: $(ncol(df_loads)) ($(join(names(df_loads)[1:min(5, end)], ", "))...)")
    println("   - Generation columns: $(ncol(df_gens)) ($(join(names(df_gens), ", ")))")
    println("   - LMP columns: $(ncol(df_lambda)) ($(join(names(df_lambda)[1:min(5, end)], ", "))...)")
    println("   - Other dual columns: $(ncol(df_mu_g_min) + ncol(df_mu_g_max) + ncol(df_mu_line_pos) + ncol(df_mu_line_neg))")
end

function main()
    ROOT_DIR = raw"C:\Users\Aloha\Desktop\dataset"
    CASE_NAME_FULL = "pglib_opf_case14_ieee"
    CASE_NAME_SHORT = "case14"
    VARIANCE_LABEL = "v=0.12"

    NUM_SAMPLES = 50000  # 先测试100个样本
    VARIANCE = 0.12
    SEED = 42
    SAVE_INTERMEDIATE_FILES = true

    case_file = joinpath(ROOT_DIR, "PGlib", "standard", "$(CASE_NAME_FULL).m")
    output_dir = joinpath(ROOT_DIR, "PINN", "$(CASE_NAME_SHORT)($(VARIANCE_LABEL))_with_nodal_lmps")

    println("=" ^ 70)
    println("🚀 生成包含每母线LMP的DCOPF数据集")
    println("=" ^ 70)
    println("输入: $(case_file)")
    println("输出: $(output_dir)")
    println("样本数: $(NUM_SAMPLES)")
    println("方差: $(VARIANCE)")
    println("=" ^ 70)
    println()

    DCOPF_dataset_with_duals(case_file, output_dir, NUM_SAMPLES, VARIANCE, SEED, save_intermediate_files=SAVE_INTERMEDIATE_FILES)
    
    println("\n" * "=" ^ 70)
    println("✅ 完成！请检查生成的CSV文件")
    println("=" ^ 70)
end

# 运行主程序
main()

🚀 生成包含每母线LMP的DCOPF数据集
输入: C:\Users\Aloha\Desktop\dataset\PGlib\standard\pglib_opf_case118_ieee.m
输出: C:\Users\Aloha\Desktop\dataset\PINN\case118(v=0.12)_with_nodal_lmps
样本数: 50000
方差: 0.12

Generating dataset for pglib_opf_case118_ieee.m
Target samples: 50000, Variance: 0.12, Seed: 42
[info | PowerModels]: removing 3 cost terms from generator 32: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 29: [3266.8781000000004, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 1: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 54: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 2: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 41: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 51: [3504.3401000000003, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 53: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 27: Float64[]
[info | PowerModels]: remov

Generating Samples 100%|█████████████████████████████████| Time: 5:46:39


Successfully generated 49532 / 50000 samples in 20799.11s.
Average cost: 93212.2645$
Average solve time: 419.91ms

📊 LMP Statistics (节点电价统计):
  Bus 1: μ ∈ [2100.4959, 2575.8442], mean=2395.6623, std=148.7838
  Bus 2: μ ∈ [2099.9618, 2575.8442], mean=2395.6403, std=148.8065
  Bus 3: μ ∈ [2100.7226, 2575.8442], mean=2395.6717, std=148.7742
  Bus 4: μ ∈ [2101.8171, 2575.8442], mean=2395.712, std=148.7317
  Bus 5: μ ∈ [2102.0357, 2575.8442], mean=2395.7258, std=148.7186
  Bus 6: μ ∈ [2100.843, 2575.8442], mean=2395.6766, std=148.7691
  Bus 7: μ ∈ [2100.3835, 2575.8442], mean=2395.6577, std=148.7886
  Bus 8: μ ∈ [2104.5053, 2575.8442], mean=2395.8613, std=148.5872
  Bus 9: μ ∈ [2104.5053, 2575.8442], mean=2395.8613, std=148.5872
  Bus 10: μ ∈ [2104.5053, 2575.8443], mean=2395.8613, std=148.5872
  Bus 11: μ ∈ [2099.9319, 2575.8442], mean=2395.5934, std=148.8443
  Bus 12: μ ∈ [2099.6325, 2575.8442], mean=2395.6268, std=148.8204
  Bus 13: μ ∈ [2096.7907, 2575.8442], mean=2395.2009, std=149.188